In [1]:
# Load the enriched Wordle lexicon and preview its first five records.
import pandas as pd

words = pd.read_json("../data/wordle-lexicon.jsonl", lines=True)
words.head(5)

,word,is_original_answer,zipf_frequency,parts_of_speech
0,aahed,False,NaN,[verb]
1,aalii,False,NaN,[noun]
2,aargh,False,1.83,[interjection]
3,aarti,False,2.23,[noun]
4,abaca,False,1.77,[noun]


In [2]:
# Summarize every column, including categorical fields.
words.describe(include="all")

,word,is_original_answer,zipf_frequency,parts_of_speech
count,12972,12972,8926.000000,12972
unique,12972,2,NaN,104
top,aahed,False,NaN,[noun]
freq,1,10657,NaN,5992
mean,NaN,NaN,2.544942,NaN
std,NaN,NaN,1.082390,NaN
min,NaN,NaN,1.010000,NaN
25%,NaN,NaN,1.650000,NaN
50%,NaN,NaN,2.355000,NaN
75%,NaN,NaN,3.260000,NaN


## 1. Shape, types, and missing values

Check the dataset dimensions, column types, missingness, cardinality, and word uniqueness before drawing conclusions from the data.

In [3]:
from IPython.display import display

display(pd.Series({
    "rows": len(words),
    "columns": len(words.columns),
    "duplicate_words": words["word"].duplicated().sum(),
}, name="dataset"))

pd.DataFrame({
    "dtype": words.dtypes.astype(str),
    "missing": words.isna().sum(),
    "missing_percent": words.isna().mean().mul(100).round(2),
    "unique_values": words.astype(str).nunique(),
})

rows               12972
columns                4
duplicate_words        0
Name: dataset, dtype: int64

,dtype,missing,missing_percent,unique_values
word,str,0,0.00,12972
is_original_answer,bool,0,0.00,2
zipf_frequency,float64,4046,31.19,484
parts_of_speech,object,0,0.00,104


## 2. Answer-set balance

Measure how much of the legal vocabulary belongs to the original answer set.

In [4]:
answer_balance = words["is_original_answer"].value_counts().sort_index()
pd.DataFrame({
    "words": answer_balance,
    "percent": answer_balance.div(len(words)).mul(100).round(2),
}).rename(index={False: "legal_guess_only", True: "original_answer"})

,words,percent
is_original_answer,,
legal_guess_only,10657,82.15
original_answer,2315,17.85


## 3. Zipf-frequency distribution

Compare the frequency range and distribution of original answers with guess-only words.

In [5]:
display(
    words.groupby("is_original_answer")["zipf_frequency"]
    .describe()
    .rename(index={False: "legal_guess_only", True: "original_answer"})
)

zipf_bins = pd.cut(words["zipf_frequency"], bins=[1, 2, 3, 4, 5, 6, 7], right=False)
pd.crosstab(zipf_bins, words["is_original_answer"]).rename(
    columns={False: "legal_guess_only", True: "original_answer"}
)

,count,mean,std,min,25%,50%,75%,max
is_original_answer,,,,,,,,
legal_guess_only,6611.0,2.175262,0.883427,1.01,1.490,1.99,2.64,5.96
original_answer,2315.0,3.600644,0.885385,1.17,2.955,3.52,4.18,6.40


is_original_answer,legal_guess_only,original_answer
zipf_frequency,,
"[1, 2)",3329,40
"[2, 3)",2177,583
"[3, 4)",777,969
"[4, 5)",289,541
"[5, 6)",39,170
"[6, 7)",0,12


## 4. Missing frequency values

Count and inspect words that have no Zipf frequency; these are likely to contain unusually obscure vocabulary.

In [6]:
missing_frequency = words[words["zipf_frequency"].isna()]
display(
    missing_frequency["is_original_answer"]
    .value_counts()
    .rename(index={False: "legal_guess_only", True: "original_answer"})
    .rename("missing_frequency_words")
)
missing_frequency.head(20)

is_original_answer
legal_guess_only    4046
Name: missing_frequency_words, dtype: int64

,word,is_original_answer,zipf_frequency,parts_of_speech
0,aahed,False,NaN,[verb]
1,aalii,False,NaN,[noun]
7,abacs,False,NaN,[noun]
9,abaka,False,NaN,[noun]
10,abamp,False,NaN,[noun]
14,abask,False,NaN,[adverb]
18,abbed,False,NaN,"[adjective, verb]"
22,abcee,False,NaN,[noun]
24,abear,False,NaN,"[noun, verb]"
34,ablet,False,NaN,[noun]


## 5. Parts-of-speech coverage

Explode multi-label classifications and compare lexical roles across answers and guess-only words.

In [7]:
pos = words.explode("parts_of_speech", ignore_index=True)
pos["parts_of_speech"] = pos["parts_of_speech"].fillna("unclassified")

pd.crosstab(pos["parts_of_speech"], pos["is_original_answer"]).rename(
    columns={False: "legal_guess_only", True: "original_answer"}
).sort_values("original_answer", ascending=False)

is_original_answer,legal_guess_only,original_answer
parts_of_speech,,
noun,8581,1929
verb,3689,1486
adjective,1479,810
adverb,181,167
interjection,146,78
preposition,33,23
determiner,10,14
proper_noun,53,12
pronoun,13,11


## 6. Letter frequencies

Count letters at each position for the complete legal vocabulary and for original answers.

In [8]:
def position_counts(frame):
    positions = frame["word"].apply(list).apply(pd.Series)
    positions.columns = [f"position_{position}" for position in range(1, 6)]
    return positions.apply(pd.Series.value_counts).fillna(0).astype(int)

pd.concat({
    "all_legal_guesses": position_counts(words),
    "original_answers": position_counts(words[words["is_original_answer"]]),
}, axis=1)

all_legal_guesses                                              \
         position_1 position_2 position_3 position_4 position_5   
a               737       2263       1236       1074        680   
b               909         81        335        243         59   
c               922        176        392        411        127   
d               685         84        390        471        823   
e               303       1628        882       2327       1522   
f               598         24        178        233         82   
g               638         76        364        423        143   
h               489        546        120        235        370   
i               165       1383       1051        880        280   
j               202         11         46         29          3   
k               376         95        272        503        259   
l               577        699        848        771        476   
m               693        188        511        402        182   
n               325        345        964        788        530   
o               262       2096        993        698        389   
p               859        231        364        418        147   
q                78         15         13          2          4   
r               628        940       1198        719        673   
s              1565         93        533        516       3958   
t               815        239        616        898        727   
u               189       1187        667        401         67   
v               242         52        240        156          4   
w               413        163        271        128         64   
x                16         57        133         12         70   
y               181        271        213        108       1301   
z               105         29        142        126         32   

  original_answers                                              
        position_1 position_2 position_3 position_4 position_5  
a              141        304        307        163         64  
b              173         16         57         24         11  
c              198         40         56        152         31  
d              111         20         75         69        118  
e               72        242        177        318        424  
f              136          8         25         35         26  
g              115         12         67         76         41  
h               69        144          9         28        139  
i               34        202        266        158         11  
j               20          2          3          2          0  
k               20         10         12         55        113  
l               88        201        112        162        156  
m              107         38         61         68         42  
n               37         87        139        182        130  
o               41        279        244        132         58  
p              142         61         58         50         56  
q               23          5          1          0          0  
r              105        267        163        152        212  
s              366         16         80        171         36  
t              149         77        111        139        253  
u               33        186        165         82          1  
v               43         15         49         46          0  
w               83         44         26         25         17  
x                0         14         12          3          8  
y                6         23         29          3        364  
z                3          2         11         20          4

## 7. Repeated-letter patterns

Compare how often words repeat letters and whether any letter occurs two or three times.

In [9]:
repeat_profile = words.assign(
    repeated_letters=words["word"].map(lambda word: 5 - len(set(word))),
    max_multiplicity=words["word"].map(
        lambda word: max(word.count(letter) for letter in set(word))
    ),
)

pd.crosstab(
    [repeat_profile["repeated_letters"], repeat_profile["max_multiplicity"]],
    repeat_profile["is_original_answer"],
).rename(columns={False: "legal_guess_only", True: "original_answer"})

is_original_answer                 legal_guess_only  original_answer
repeated_letters max_multiplicity                                   
0                1                             6756             1566
1                2                             3494              691
2                2                              297               38
                 3                              102               19
3                3                                8                1

## 8. Word structure

Measure unique letters and vowels, then inspect common two-letter prefixes and suffixes.

In [10]:
structure = words.assign(
    unique_letters=words["word"].map(lambda word: len(set(word))),
    vowel_count=words["word"].map(lambda word: sum(letter in "aeiou" for letter in word)),
    prefix=words["word"].str[:2],
    suffix=words["word"].str[-2:],
)

display(
    structure.groupby("is_original_answer")[["unique_letters", "vowel_count"]]
    .mean()
    .round(2)
    .rename(index={False: "legal_guess_only", True: "original_answer"})
)

pd.DataFrame({
    "common_prefixes": structure["prefix"].value_counts().head(15),
    "common_suffixes": structure["suffix"].value_counts().head(15),
})

,unique_letters,vowel_count
is_original_answer,,
legal_guess_only,4.60,1.81
original_answer,4.65,1.77


,common_prefixes,common_suffixes
al,NaN,168.0
as,NaN,292.0
ba,166.0,NaN
bo,175.0,NaN
ca,168.0,NaN
ch,147.0,NaN
co,220.0,NaN
do,153.0,NaN
ds,NaN,211.0
ed,NaN,530.0


## 9. Frequency outliers

Inspect the least and most familiar words with recorded frequencies, retaining answer status and parts of speech.

In [11]:
recorded_frequency = words.dropna(subset=["zipf_frequency"])
pd.concat([
    recorded_frequency.nsmallest(15, "zipf_frequency").assign(extreme="lowest"),
    recorded_frequency.nlargest(15, "zipf_frequency").assign(extreme="highest"),
])[['extreme', 'word', 'zipf_frequency', 'is_original_answer', 'parts_of_speech']]

,extreme,word,zipf_frequency,is_original_answer,parts_of_speech
227,lowest,airer,1.01,False,[noun]
522,lowest,argol,1.01,False,[noun]
852,lowest,basen,1.01,False,[verb]
871,lowest,basts,1.01,False,[noun]
1839,lowest,cepes,1.01,False,[noun]
2725,lowest,deked,1.01,False,[verb]
3297,lowest,educe,1.01,False,"[noun, verb]"
3695,lowest,femmy,1.01,False,[adjective]
4187,lowest,galut,1.01,False,[noun]
4224,lowest,gares,1.01,False,[noun]


## 10. Answer-selection bias

Combine the main indicators into one comparison of original answers and guess-only vocabulary.

In [12]:
selection_bias = words.assign(
    frequency_recorded=words["zipf_frequency"].notna(),
    unique_letters=words["word"].map(lambda word: len(set(word))),
    vowel_count=words["word"].map(lambda word: sum(letter in "aeiou" for letter in word)),
    has_repeated_letter=words["word"].map(lambda word: len(set(word)) < 5),
    classified=words["parts_of_speech"].map(bool),
)

selection_bias.groupby("is_original_answer").agg(
    words=("word", "size"),
    mean_zipf=("zipf_frequency", "mean"),
    frequency_coverage=("frequency_recorded", "mean"),
    classification_coverage=("classified", "mean"),
    mean_unique_letters=("unique_letters", "mean"),
    mean_vowels=("vowel_count", "mean"),
    repeated_letter_rate=("has_repeated_letter", "mean"),
).rename(index={False: "legal_guess_only", True: "original_answer"}).round(3)

,words,mean_zipf,frequency_coverage,classification_coverage,mean_unique_letters,mean_vowels,repeated_letter_rate
is_original_answer,,,,,,,
legal_guess_only,10657,2.175,0.62,1.0,4.595,1.807,0.366
original_answer,2315,3.601,1.00,1.0,4.651,1.773,0.324
